rete neurale con un solo strato nascosto e albero di regressione con prima una riduzione della dimensionalità fatta a scelta

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()

X, y = housing.data, housing.target

print(housing.data.shape, housing.target.shape)
print(housing.feature_names)

print('Target names:', housing.target_names)
print('Number of targets:', np.unique(y))

(20640, 8) (20640,)
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Target names: ['MedHouseVal']
Number of targets: [0.14999 0.175   0.225   ... 4.991   5.      5.00001]


## Nested CV

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, r2_score, root_mean_squared_error, mean_absolute_error

def nested_cv(model, param_grid, X, y, outer_splits=5,
              inner_splits=5, scoring=['accuracy', 'roc_auc'],
              random_state=42, verbose=True):

    # Assicurati che `y` sia un array 1D
    if isinstance(y, pd.DataFrame):  # Se è un DataFrame Pandas
        y = y.values.ravel()
    elif isinstance(y, pd.Series):  # Se è una Serie Pandas
        y = y.values
    else:  # Se è un array Numpy
        y = np.ravel(y)

    # CROSS-VALIDATION ESTERNA PER ITERARE SUI DIVERSI TEST SET
    outer_cv = KFold(n_splits=outer_splits, shuffle=True, random_state=random_state)
    score_results = {metric: [] for metric in scoring}

    best_param_overall = None
    main_metric = scoring[0]

    # 1. INIZIALIZZAZIONE INTELLIGENTE
    # Se la metrica è un errore, partiamo da infinito per minimizzare.
    # Altrimenti, partiamo da -infinito per massimizzare.
    if main_metric in ['mae', 'rmse']:
        best_score = np.inf
    else:
        best_score = -np.inf

    # Logica di conversione
    main_metric_for_gridsearch = main_metric
    if main_metric == 'mae':
        main_metric_for_gridsearch = 'neg_mean_absolute_error'
    elif main_metric == 'rmse':
        # Nelle versioni recenti di sklearn, 'rmse' è un alias per 'neg_root_mean_squared_error',
        # ma essere espliciti è più sicuro.
        main_metric_for_gridsearch = 'neg_root_mean_squared_error'

    for outer_fold, (train_idx, test_idx) in enumerate(outer_cv.split(X), 1):
        if verbose:
            print(f"\nPerforming Outer Fold {outer_fold}/{outer_splits}")

        # Usare il metodo .iloc per X, se è un DataFrame
        if isinstance(X, pd.DataFrame):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        else:  # Altrimenti usa indicizzazione standard
            X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # CROSS-VALIDATION INTERNA con GridSearchCV PER TROVARE I MIGLIORI IPER-PARAMETRI SU QUEL TRAINING SET
        inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=random_state)
        if verbose:
            print("Performing GridSearchCV...")

        # Usa la metrica convertita in GridSearchCV
        grid_search = GridSearchCV(model, param_grid, cv=inner_cv,
                                   n_jobs=-1, scoring=main_metric_for_gridsearch)
        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_

        if verbose:
            print(f" Best Params: {best_params}")

        # Test del modello ottimale sui dati di test dell'outer fold
        y_pred = best_model.predict(X_test)

        # Calcolo delle metriche
        if 'accuracy' in scoring:
            acc = accuracy_score(y_test, y_pred)
            score_results['accuracy'].append(acc)
            if main_metric == 'accuracy' and acc > best_score:
                best_score = acc
                best_param_overall = best_params
            if verbose:
                print(f" Accuracy: {acc:.4f}")

        if 'roc_auc' in scoring:
            try:
                y_score = best_model.predict_proba(X_test)[:, 1]
                auc = roc_auc_score(y_test, y_score)
                score_results['roc_auc'].append(auc)
                if main_metric == 'roc_auc' and auc > best_score:
                    best_score = auc
                    best_param_overall = best_params
                if verbose:
                    print(f" AUC: {auc:.4f}")
            except AttributeError:
                if verbose:
                    print("Controlla se il modello ha un metodo `predict_proba`.")
                score_results['roc_auc'].append(np.nan)

        if 'r2' in scoring:
            r2score = r2_score(y_test, y_pred)
            score_results['r2'].append(r2score)
            if main_metric == 'r2' and r2score > best_score:
                best_score = r2score
                best_param_overall = best_params
            if verbose:
                print(f" R2: {r2score:.4f}")

        if 'mae' in scoring:
            mae = mean_absolute_error(y_test, y_pred)
            score_results['mae'].append(mae)
            if main_metric == 'mae' and (best_score == -np.inf or mae < best_score):
                best_score = mae
                best_param_overall = best_params
            if verbose:
                print(f" MAE: {mae:.4f}")

        if 'rmse' in scoring:
            rmse = root_mean_squared_error(y_test, y_pred)
            score_results['rmse'].append(rmse)
            if main_metric == 'rmse' and (best_score == -np.inf or rmse < best_score):
                best_score = rmse
                best_param_overall = best_params
            if verbose:
                print(f" RMSE: {rmse:.4f}")

    result = {}
    for metric, scores in score_results.items():
        result[f"Nested CV {metric.upper()}"] = f"{np.nanmean(scores):.4f} ± {np.nanstd(scores):.4f}"

    result["Best Parameters with best " + main_metric] = best_param_overall

    return result

## MLP con un solo strato nascosto

In [3]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

mlp_pipeline = Pipeline([
    ('pca', PCA()),
    ('mlp', MLPRegressor())
])

In [4]:
mlp_params_grid = {
    'pca__n_components': [2, 3, 5],
    'pca__svd_solver': ['full', 'randomized'],
    'mlp__hidden_layer_sizes': [(3,)],
    'mlp__activation': ['logistic', 'relu']
}

In [5]:
mlp_result = nested_cv(model = mlp_pipeline,
                       X = X,
                       y = y,
                       param_grid = mlp_params_grid,
                       scoring = ['r2'])


Performing Outer Fold 1/5
Performing GridSearchCV...
 Best Params: {'mlp__activation': 'logistic', 'mlp__hidden_layer_sizes': (3,), 'pca__n_components': 5, 'pca__svd_solver': 'full'}
 R2: 0.4262

Performing Outer Fold 2/5
Performing GridSearchCV...
 Best Params: {'mlp__activation': 'logistic', 'mlp__hidden_layer_sizes': (3,), 'pca__n_components': 5, 'pca__svd_solver': 'randomized'}
 R2: 0.4373

Performing Outer Fold 3/5
Performing GridSearchCV...
 Best Params: {'mlp__activation': 'logistic', 'mlp__hidden_layer_sizes': (3,), 'pca__n_components': 5, 'pca__svd_solver': 'randomized'}
 R2: 0.4058

Performing Outer Fold 4/5
Performing GridSearchCV...
 Best Params: {'mlp__activation': 'logistic', 'mlp__hidden_layer_sizes': (3,), 'pca__n_components': 5, 'pca__svd_solver': 'full'}
 R2: 0.4320

Performing Outer Fold 5/5
Performing GridSearchCV...
 Best Params: {'mlp__activation': 'logistic', 'mlp__hidden_layer_sizes': (3,), 'pca__n_components': 5, 'pca__svd_solver': 'full'}
 R2: 0.4358


## Decision Tree Regressor

In [6]:
from sklearn.tree import DecisionTreeRegressor

dt_regressor_pipeline = Pipeline([
    ('pca', PCA()),
    ('regressor', DecisionTreeRegressor())
])

In [7]:
dt_regressor_params_grid = {
    'pca__n_components': [2, 3, 5],
    'pca__svd_solver': ['full', 'randomized'],
    'regressor__splitter': ['best', 'random'],
    'regressor__max_depth': [5, 10, 20]
}

In [8]:
dt_regressor_result = nested_cv(model = dt_regressor_pipeline,
                       X = X,
                       y = y,
                       param_grid = dt_regressor_params_grid,
                       scoring = ['r2'])


Performing Outer Fold 1/5
Performing GridSearchCV...
 Best Params: {'pca__n_components': 5, 'pca__svd_solver': 'full', 'regressor__max_depth': 10, 'regressor__splitter': 'best'}
 R2: 0.4039

Performing Outer Fold 2/5
Performing GridSearchCV...
 Best Params: {'pca__n_components': 5, 'pca__svd_solver': 'randomized', 'regressor__max_depth': 10, 'regressor__splitter': 'best'}
 R2: 0.3987

Performing Outer Fold 3/5
Performing GridSearchCV...
 Best Params: {'pca__n_components': 5, 'pca__svd_solver': 'randomized', 'regressor__max_depth': 10, 'regressor__splitter': 'best'}
 R2: 0.3855

Performing Outer Fold 4/5
Performing GridSearchCV...
 Best Params: {'pca__n_components': 5, 'pca__svd_solver': 'full', 'regressor__max_depth': 10, 'regressor__splitter': 'best'}
 R2: 0.4717

Performing Outer Fold 5/5
Performing GridSearchCV...
 Best Params: {'pca__n_components': 5, 'pca__svd_solver': 'full', 'regressor__max_depth': 5, 'regressor__splitter': 'best'}
 R2: 0.4159


## Random Forest Regressor

In [9]:
from sklearn.ensemble import RandomForestRegressor

forest_pipeline = Pipeline([
    ('pca', PCA()),
    ('forest', RandomForestRegressor())
])

In [10]:
forest_params_grid = {
    'pca__n_components': [2, 3, 5],
    'pca__svd_solver': ['full', 'randomized'],
    'forest__n_estimators': [50, 100, 150],
    'forest__max_depth': [None, 10, 20]
}

In [11]:
forest_result = nested_cv(model = forest_pipeline,
                       X = X,
                       y = y,
                       param_grid = forest_params_grid,
                       scoring = ['r2'])


Performing Outer Fold 1/5
Performing GridSearchCV...
 Best Params: {'forest__max_depth': None, 'forest__n_estimators': 150, 'pca__n_components': 5, 'pca__svd_solver': 'full'}
 R2: 0.5480

Performing Outer Fold 2/5
Performing GridSearchCV...
 Best Params: {'forest__max_depth': 20, 'forest__n_estimators': 150, 'pca__n_components': 5, 'pca__svd_solver': 'randomized'}
 R2: 0.5505

Performing Outer Fold 3/5
Performing GridSearchCV...
 Best Params: {'forest__max_depth': None, 'forest__n_estimators': 150, 'pca__n_components': 5, 'pca__svd_solver': 'full'}
 R2: 0.5123

Performing Outer Fold 4/5
Performing GridSearchCV...
 Best Params: {'forest__max_depth': None, 'forest__n_estimators': 150, 'pca__n_components': 5, 'pca__svd_solver': 'randomized'}
 R2: 0.6109

Performing Outer Fold 5/5
Performing GridSearchCV...
 Best Params: {'forest__max_depth': None, 'forest__n_estimators': 150, 'pca__n_components': 5, 'pca__svd_solver': 'full'}
 R2: 0.5751


## Teoria
- Differenza tra Random Forest e Decision Tree
- Perchè nel Random Forest si usa il rimpiazzo e non la suddivisione in diversi training set totalmente disgiunti a coppie (mutuamente esclusivi): Poichè a noi interessa la diversità se abbiamo un dataset con 500 esempi, ogni albero verrebbe addestrato su 5 esempi (il numero di default di sklearn per il numero di alberi è 100).
- L'ulteriore differenza della foresta rispetto al bagging: la random feature selection per assicurare una certa diversità
- Particolare forma non parametrica della PCA che lavora meglio per alcuni dataset: la Kernel PCA
- Su quale matrice lavora la PCA? La matrice di covarianza $\Sigma = X^T X$
- Su quale matrice lavora la KPCA? Su $K = < \phi(X), \phi(X') >$. I vettori di partenza una volta trasformati formano una matrice (hanno il loro embedding nello spazio del kernel)
- Come si passa da SVM ad SVR: cambiano le metriche e poi cerchiamo un tubo all'interno del quale non consideriamo l'errore. Questo si trasforma in una funzione dove l'errore vale 0 quando $|\hat{y} - y | < \varepsilon$, altrimenti $|\hat{y} - y| - \varepsilon|$ (variabile di slack)
- Se siamo non nella forma kernelizzata il tubo si concentra attorno all'iperpiano separatore
- Nel caso dell'SVR quali sono i vettori di supporto? Nella forma soft quelli che si trovano al di fuori dal tubo
- alpha (moltiplicatori di lagrange) non nulli per i vettori di supporto. In generale mi devo ricordare dei coefficienti sbagliati (previsioni sbagliate)
- La complessità dell'SVM dipende da N (cardinalità dataset) --> differenza tra modelli parametrici e non parametrici
- Poichè solitamente N è solitamente maggiore di D i modelli non parametrici che dipendono da N potrebbero sembrare svantaggiosi. Ma in realtà dipendendendo da pochi parametri (Vettori di supporto) soddisfano la proprietà di dipendere da pochi parametri: soluzione sparsa